# Human-Like Move Picker — Training Notebook

This notebook trains a **supervised model** (no RL) that:

1. **Chooses a human-like move** among Stockfish top-20 candidates (XGBoost Ranker)
2. **Predicts think time** for the chosen move (XGBoost Regressor)

**Inputs per position:** top-20 Stockfish moves + scores, human prior (freq/mean_ms) from your DB-derived prior, basic context (side, ply, legal_count).  
**Outputs:** `picked_uci`, `pred_think_ms`.

> ⚙️ Assumes you already have:
> - `ENGINE_PATH` pointing to your Stockfish exe (Windows .exe path)
> - `HUMAN_PRIOR_PATH` pointing to the 2200–2400 JSON prior you built
> - Azure SQL credentials to read `dbo.game_core` + `dbo.game_text`


## 0) Env setup (run once per kernel)

In [ ]:
# If using conda 'chess_env', pip installs go into that env.
!pip -q install xgboost pandas pyarrow scikit-learn python-dotenv python-chess pyodbc
print('✅ Dependencies installed')

## 1) Imports + utility paths

In [ ]:
import os, io, re, sys, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error
import pyodbc
import chess, chess.pgn
from dotenv import load_dotenv

# Make local package imports work if this notebook runs from repo root
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

load_dotenv()  # loads .env from repo root

print('CWD =', os.getcwd())
print('ENGINE_PATH =', os.getenv('ENGINE_PATH'))
print('HUMAN_PRIOR_PATH =', os.getenv('HUMAN_PRIOR_PATH'))

## 2) Engine & Prior sanity checks

In [ ]:
from chess_bot.engine.stockfish_service import StockfishService, SHORTLIST_N
from chess_bot.policy.human_prior_store import HumanPriorStore

svc = StockfishService(); svc.open()
store = HumanPriorStore()

start_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
top = svc.get_top_moves(start_fen, n=SHORTLIST_N)
prior = store.get_prior(start_fen)
print('Top moves:', [m['uci'] for m in top['top_moves'][:5]])
print('Prior sample:', prior['moves'][:5])
svc.close()

## 3) Candidate builder (verbose)
Streams games from Azure SQL, parses plies with clocks, collects top-20 and prior features.

**Environment variables expected:**
- `AZURE_SQL_SERVER`, `AZURE_SQL_DB`, `AZURE_SQL_USER`, `AZURE_SQL_PASSWORD`, `AZURE_SQL_DRIVER`
- `PRIOR_ELO_MIN`, `PRIOR_ELO_MAX`
- `CANDIDATE_MAX_POS`, `CANDIDATE_MAX_PLY`, `PRINT_EVERY`


In [ ]:
TC_RE  = re.compile(r"^\s*(\d+)(?:\+(\d+))?\s*$")
CLK_RE = re.compile(r"\[\s*%clk\s+([0-9:]+)\s*\]")

def clock_to_ms(clk: str):
    if not clk: return None
    parts = [int(p) for p in clk.split(":")]
    if len(parts)==2: h,m,s = 0, parts[0], parts[1]
    elif len(parts)==3: h,m,s = parts
    else: return None
    return ((h*60+m)*60+s)*1000

def parse_timecontrol(tc: str):
    if not tc: return None, None
    m = TC_RE.match(tc.strip()); 
    if not m: return None, None
    base = int(m.group(1))*1000; inc = int(m.group(2) or 0)*1000
    return base, inc

def pgntxt(start_fen, movetext):
    headers = ['[Event "-"]','[Site "-"]','[Date "????.??.??"]','[Round "-"]',
               '[White "-"]','[Black "-"]','[Result "*"]','[SetUp "1"]', f'[FEN "{start_fen}"]']
    body = movetext.strip()
    if not body.endswith(("1-0","0-1","1/2-1/2","*")): body += " *"
    return "\n".join(headers) + "\n\n" + body + "\n"

def iter_plies(start_fen: str, movetext_full: str, timecontrol: str):
    base_ms, inc_ms = parse_timecontrol(timecontrol or "")
    game = chess.pgn.read_game(io.StringIO(pgntxt(start_fen, movetext_full)))
    if not game: return
    board = game.board()
    prev_post = {True: None, False: None}
    ply = 0
    for node in game.mainline():
        if node.move is None: 
            continue
        fen_before = board.fen()
        side = board.turn
        uci = node.move.uci()
        san = board.san(node.move)
        board.push(node.move)
        ply += 1

        post_ms = None
        if node.comment:
            m = CLK_RE.search(node.comment)
            if m: post_ms = clock_to_ms(m.group(1))

        think_ms = None
        if post_ms is not None and base_ms is not None:
            pre = base_ms if prev_post[side] is None else max(prev_post[side] + (inc_ms or 0), 0)
            d = pre - post_ms
            if 0 <= d <= 10*60*1000:
                think_ms = d

        prev_post[side] = post_ms

        yield {"ply": ply, "fen": fen_before, "side": "w" if side else "b",
               "human_uci": uci, "human_san": san, "think_ms": think_ms}

def connect_sql():
    SQL_SERVER   = os.getenv("AZURE_SQL_SERVER")
    SQL_DB       = os.getenv("AZURE_SQL_DB")
    SQL_USER     = os.getenv("AZURE_SQL_USER")
    SQL_PASSWORD = os.getenv("AZURE_SQL_PASSWORD")
    SQL_DRIVER   = os.getenv("AZURE_SQL_DRIVER", "ODBC Driver 18 for SQL Server")

    cs = (
        f"DRIVER={{{SQL_DRIVER}}};SERVER={SQL_SERVER};DATABASE={SQL_DB};UID={SQL_USER};PWD={SQL_PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    print(f"[SQL] Connecting to {SQL_SERVER}/{SQL_DB} with {SQL_DRIVER} ...")
    return pyodbc.connect(cs)

def build_candidates(max_positions=2000, keep_max_ply=60, print_every=200, out_parquet="data/candidates_2200_2400.parquet"):
    from chess_bot.engine.stockfish_service import StockfishService, SHORTLIST_N
    from chess_bot.policy.human_prior_store import HumanPriorStore

    ELO_MIN = int(os.getenv("PRIOR_ELO_MIN", "2200"))
    ELO_MAX = int(os.getenv("PRIOR_ELO_MAX", "2400"))

    t0 = time.time()
    Path("data").mkdir(exist_ok=True, parents=True)

    print(f"[SETUP] ELO filter: {ELO_MIN}-{ELO_MAX}, MAX_POSITIONS={max_positions}, KEEP_MAX_PLY={keep_max_ply}")
    print(f"[SETUP] Loading HumanPriorStore from ENV (HUMAN_PRIOR_PATH) ...")
    store = HumanPriorStore(); print("[SETUP] HumanPriorStore loaded.")

    print(f"[SETUP] Opening StockfishService (SHORTLIST_N={SHORTLIST_N}) ...")
    svc = StockfishService(); svc.open(); print(f"[SETUP] StockfishService ready.")

    with connect_sql() as conn, conn.cursor() as cur:
        print("[SQL] Executing query for candidate games ...")
        cur.execute(f"""
            SELECT TOP {max_positions*2}
              core.game_pk,
              COALESCE(NULLIF(core.start_fen,''),'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1') AS start_fen,
              COALESCE(core.timecontrol,'') AS timecontrol,
              txt.pgn_movetext_full
            FROM dbo.game_core core
            JOIN dbo.game_text txt ON txt.game_pk = core.game_pk
            WHERE txt.pgn_movetext_full IS NOT NULL
              AND core.elo_avg BETWEEN ? AND ?
            ORDER BY core.game_pk;
        """, (ELO_MIN, ELO_MAX))
        rows = cur.fetchall()
        print(f"[SQL] Retrieved {len(rows)} games (pre-scan).")

    recs = []
    kept_positions = 0
    positions_scanned = 0
    prior_hits = 0
    clock_rows = 0

    for ridx, r in enumerate(rows, 1):
        start_fen, tc, movetext = r.start_fen, r.timecontrol, r.pgn_movetext_full
        if ridx % max(1, print_every//10) == 0:
            print(f"[SCAN] Game #{ridx} (kept_positions={kept_positions}) ...")
        try:
            for pl in iter_plies(start_fen, movetext, tc):
                if pl["ply"] > keep_max_ply: break
                positions_scanned += 1
                fen, side, human_uci, think_ms = pl["fen"], pl["side"], pl["human_uci"], pl["think_ms"]
                if think_ms is not None: clock_rows += 1

                eng = svc.get_top_moves(fen, n=SHORTLIST_N)["top_moves"]
                if not eng: continue
                uci_list = [m["uci"] for m in eng]
                if human_uci not in uci_list:
                    continue

                prior = store.get_prior(fen)
                if prior.get("moves"): prior_hits += 1
                freq_map = {m["uci"]: m["freq"] for m in prior.get("moves", [])}
                mean_map = {m["uci"]: m.get("mean_ms") for m in prior.get("moves", [])}
                total_freq = prior.get("total", 0) or 1

                group_id = f"{hash(fen)}:{kept_positions}"

                cp_vals = [(-10**9 if m.get("score_cp") is None else m["score_cp"]) for m in eng]
                cp_arr = np.array(cp_vals, dtype=np.float32)
                tau = 60.0
                exps = np.exp((cp_arr/tau) - (np.max(cp_arr)/tau))
                p_engine = (exps / (exps.sum() or 1.0)).tolist()
                best_cp = max([m["score_cp"] for m in eng if m.get("score_cp") is not None], default=None)

                for i, m in enumerate(eng):
                    uci = m["uci"]
                    recs.append({
                        "group_id": group_id,
                        "fen": fen,
                        "side": 1 if side=="w" else 0,
                        "uci": uci,
                        "label_choice": 1 if uci==human_uci else 0,

                        "cp": m.get("score_cp"),
                        "mate": m.get("mate"),
                        "depth": m.get("depth"),
                        "pv_len": len(m.get("pv","").split()) if m.get("pv") else 0,
                        "engine_soft_p": p_engine[i],

                        "prior_freq": freq_map.get(uci, 0),
                        "prior_prob": (freq_map.get(uci, 0)/total_freq),
                        "prior_mean_ms": mean_map.get(uci),

                        "human_think_ms": think_ms,
                        "ply": pl["ply"],
                        "legal_count": len(uci_list),
                        "cp_to_best": (0 if (best_cp is None or m.get("score_cp") is None) else best_cp - m["score_cp"]),
                    })

                kept_positions += 1
                if kept_positions % print_every == 0:
                    print(f"[KEEP] kept_positions={kept_positions} | rows={len(recs)} | prior_hits={prior_hits} | clock_rows={clock_rows}")
                if kept_positions >= max_positions:
                    print("[STOP] Reached MAX_POSITIONS cap.")
                    raise StopIteration
        except StopIteration:
            break

    df = pd.DataFrame.from_records(recs)
    Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_parquet, index=False)
    dt = time.time() - t0
    print("\n========= SUMMARY =========")
    print(f"time: {dt:.1f}s")
    print(f"kept_positions: {kept_positions}")
    print(f"rows_written: {len(df)} (≈ {len(df)/max(1,kept_positions):.1f} per pos)")
    print(f"prior_hits: {prior_hits}")
    print(f"clock_rows (think_ms present): {clock_rows}")
    print(f"out: {out_parquet}")
    print("===========================")
    return df


### 3a) Run candidate build (start small to verify)

In [ ]:
df = build_candidates(max_positions=int(os.getenv('CANDIDATE_MAX_POS', '1000')),
                      keep_max_ply=int(os.getenv('CANDIDATE_MAX_PLY', '60')),
                      print_every=int(os.getenv('PRINT_EVERY', '200')),
                      out_parquet=os.getenv('CANDIDATE_OUT', 'data/candidates_2200_2400.parquet'))
df.head(10)

## 4) Train XGBoost Ranker (choice) + Regressor (time)

In [ ]:
DATA_PATH = os.getenv('CANDIDATE_OUT', 'data/candidates_2200_2400.parquet')
MODEL_DIR = Path(os.getenv('MODEL_OUT', 'models/human_pick'))
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
grp_has_pos = df.groupby('group_id')['label_choice'].max()
keep_groups = grp_has_pos[grp_has_pos > 0].index
df = df[df['group_id'].isin(keep_groups)].reset_index(drop=True)
print('Rows after enforcing human-in-top20:', len(df))

feat_cols = [
    'side','cp','mate','depth','pv_len','engine_soft_p',
    'prior_freq','prior_prob','prior_mean_ms',
    'ply','legal_count','cp_to_best'
]
for c in feat_cols:
    df[c] = df[c].fillna(0)

groups = df['group_id'].unique()
train_g, val_g = train_test_split(groups, test_size=0.15, random_state=42)
train = df[df.group_id.isin(train_g)].copy()
val   = df[df.group_id.isin(val_g)].copy()

def make_dm(d):
    X = d[feat_cols].astype(np.float32).values
    y = d['label_choice'].astype(np.float32).values
    gsizes = d.groupby('group_id')['uci'].count().to_list()
    dm = xgb.DMatrix(X, label=y)
    dm.set_group(gsizes)
    return dm

dtrain = make_dm(train); dval = make_dm(val)
params = dict(objective='rank:pairwise', eval_metric='ndcg@1', tree_method='hist',
              max_depth=6, eta=0.08, subsample=0.9, colsample_bytree=0.9, min_child_weight=10)

rk = xgb.train(params=params, dtrain=dtrain, num_boost_round=800,
               evals=[(dtrain,'train'),(dval,'val')], early_stopping_rounds=50, verbose_eval=50)
rk.save_model(str(MODEL_DIR / 'xgb_ranker.json'))

# Top-1 match on val
val['util'] = rk.predict(xgb.DMatrix(val[feat_cols].astype(np.float32).values))
pred_idx = val.groupby('group_id')['util'].idxmax(); pred = val.loc[pred_idx]
acc_top1 = pred['label_choice'].mean()
print(f"Top-1 human match (val): {acc_top1:.3f}")

# Time regressor on human-chosen rows with think_ms
time_rows = df[(df['label_choice']==1) & df['human_think_ms'].notna()].copy()
if len(time_rows) > 1000:
    time_rows['log_time'] = np.log1p(time_rows['human_think_ms'].clip(lower=0))
    time_feats = feat_cols  # reuse
    Xtr, Xte, ytr, yte = train_test_split(time_rows[time_feats].astype(np.float32).values,
                                          time_rows['log_time'].values, test_size=0.15, random_state=42)
    dtr = xgb.DMatrix(Xtr, label=ytr); dte = xgb.DMatrix(Xte, label=yte)
    rt = xgb.train(params=dict(objective='reg:squarederror', eval_metric='rmse', max_depth=6, eta=0.08,
                               tree_method='hist', subsample=0.9, colsample_bytree=0.9, min_child_weight=10),
                   dtrain=dtr, num_boost_round=600, evals=[(dtr,'train'),(dte,'val')],
                   early_stopping_rounds=50, verbose_eval=50)
    rt.save_model(str(MODEL_DIR / 'xgb_time.json'))
    pred_log = rt.predict(dte)
    mae_s = mean_absolute_error(np.expm1(yte)/1000.0, np.expm1(pred_log)/1000.0)
    print(f"Time MAE (seconds): {mae_s:.2f}")
    json.dump({'feat_cols': feat_cols, 'time_feat_cols': time_feats}, open(MODEL_DIR/'meta.json','w'))
else:
    print('Not enough rows with think_ms to train time regressor; saving ranker only.')
    json.dump({'feat_cols': feat_cols, 'time_feat_cols': None}, open(MODEL_DIR/'meta.json','w'))

print('✅ Models saved to', MODEL_DIR)

## 5) Inference — pick move + predict time

In [ ]:
from chess_bot.engine.stockfish_service import StockfishService, SHORTLIST_N
from chess_bot.policy.human_prior_store import HumanPriorStore

MODEL_DIR = Path(os.getenv('MODEL_OUT', 'models/human_pick'))
meta = json.load(open(MODEL_DIR/'meta.json'))
FEATS = meta['feat_cols']; TIME_FEATS = meta.get('time_feat_cols')

rk = xgb.Booster(); rk.load_model(str(MODEL_DIR/'xgb_ranker.json'))
rt = None
if TIME_FEATS and (MODEL_DIR/'xgb_time.json').exists():
    rt = xgb.Booster(); rt.load_model(str(MODEL_DIR/'xgb_time.json'))

store = HumanPriorStore(); svc = StockfishService(); svc.open()

def build_rows_for_fen(fen: str):
    eng = svc.get_top_moves(fen, n=SHORTLIST_N)['top_moves']
    if not eng: return [], []
    uci_list = [m['uci'] for m in eng]
    best_cp = max([m['score_cp'] for m in eng if m.get('score_cp') is not None], default=None)
    cp_list = [(-10**9 if m.get('score_cp') is None else m['score_cp']) for m in eng]
    tau = 60.0
    exps = np.exp((np.array(cp_list)/tau) - (np.max(np.array(cp_list))/tau))
    pe = (exps / (exps.sum() or 1.0)).tolist()

    prior = store.get_prior(fen)
    freq_map = {m['uci']: m['freq'] for m in prior.get('moves', [])}
    mean_map = {m['uci']: m.get('mean_ms') for m in prior.get('moves', [])}
    total = prior.get('total', 0) or 1
    prior_probs = [(freq_map.get(u,0)/total) for u in uci_list]

    board = chess.Board(fen); side = 1 if board.turn else 0
    legal_count = len(uci_list); ply = board.fullmove_number*2 - (0 if board.turn else 1)

    rows = []
    for i, m in enumerate(eng):
        rows.append(dict(
            side=side, cp=(m.get('score_cp') or 0), mate=(m.get('mate') or 0),
            depth=(m.get('depth') or 0), pv_len=len(m.get('pv','').split()) if m.get('pv') else 0,
            engine_soft_p=pe[i],
            prior_freq=freq_map.get(m['uci'], 0), prior_prob=prior_probs[i], prior_mean_ms=(mean_map.get(m['uci']) or 0),
            ply=ply, legal_count=legal_count, cp_to_best=(0 if (best_cp is None or m.get('score_cp') is None) else best_cp - m['score_cp']),
        ))
    return rows, uci_list

def pick_move(fen: str):
    rows, uci_list = build_rows_for_fen(fen)
    if not rows:
        return None
    X = np.array([[r[c] for c in FEATS] for r in rows], dtype=np.float32)
    util = rk.predict(xgb.DMatrix(X))
    idx = int(np.argmax(util))
    picked = uci_list[idx]
    think_ms = None
    if rt is not None and TIME_FEATS:
        x_time = np.array([[rows[idx][c] for c in TIME_FEATS]], dtype=np.float32)
        log_ms = rt.predict(xgb.DMatrix(x_time))[0]
        think_ms = int(np.expm1(log_ms))
    return dict(fen=fen, picked=picked, util=float(util[idx]), pred_think_ms=think_ms)

fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
res = pick_move(fen)
svc.close()
res

## 6) Notes / Next steps
- Increase `CANDIDATE_MAX_POS` once the small run works and metrics look sane.
- You can enrich features (captures, promotions, checks) to boost Top-1.
- Add a **hard safety** pre-check at inference: take +mate, avoid −mate.
- If you want a single neural model instead of XGB, swap ranker for a small MLP with softmax-over-20 and a time head.